In [97]:
torch.manual_seed(42)
inputs = torch.randn(6, 3)
inputs

tensor([[ 1.9269,  1.4873, -0.4974],
        [ 0.4396, -0.7581,  1.0783],
        [ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]])

In [98]:
import torch

inputs = torch.tensor(
  [[ 1.9269,  1.4873, -0.4974],
    [ 0.4396, -0.7581,  1.0783],
    [ 0.8008,  1.6806,  0.3559],
    [-0.6866,  0.6105,  1.3347],
    [-0.2316,  0.0418, -0.2516],
    [ 0.8599, -0.3097, -0.3957]]
)

In [99]:
x_2 = inputs[1]
d_out = 2
d_in = inputs.shape[1]

In [100]:
torch.manual_seed(42)
Wq = torch.nn.Linear(d_in,d_out,bias = False)
Wk = torch.nn.Linear(d_in,d_out,bias = False)
Wv = torch.nn.Linear(d_in,d_out,bias = False)

In [101]:
Wq.weight

Parameter containing:
tensor([[ 0.4414,  0.4792, -0.1353],
        [ 0.5304, -0.1265,  0.1165]], requires_grad=True)

In [102]:
Wk.weight

Parameter containing:
tensor([[-0.2811,  0.3391,  0.5090],
        [-0.4236,  0.5018,  0.1081]], requires_grad=True)

In [103]:
Wv.weight

Parameter containing:
tensor([[ 0.4266,  0.0782,  0.2784],
        [-0.0815,  0.4451,  0.0853]], requires_grad=True)

In [104]:
print(x_2.shape)
print(Wq.weight.shape)

torch.Size([3])
torch.Size([2, 3])


In [105]:
query_2 = x_2 @ Wq.weight.T
key_2 = x_2 @ Wk.weight.T
value_2 = x_2 @ Wv.weight.T
print(query_2)

tensor([-0.3151,  0.4547], grad_fn=<SqueezeBackward4>)


In [106]:
keys = inputs @ Wk.weight.T
queries = inputs @ Wq.weight.T
values = inputs @ Wv.weight.T

In [107]:
inputs

tensor([[ 1.9269,  1.4873, -0.4974],
        [ 0.4396, -0.7581,  1.0783],
        [ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]])

In [108]:
print(Wq.weight.T)
print(Wk.weight.T)
print(Wv.weight.T)

tensor([[ 0.4414,  0.5304],
        [ 0.4792, -0.1265],
        [-0.1353,  0.1165]], grad_fn=<PermuteBackward0>)
tensor([[-0.2811, -0.4236],
        [ 0.3391,  0.5018],
        [ 0.5090,  0.1081]], grad_fn=<PermuteBackward0>)
tensor([[ 0.4266, -0.0815],
        [ 0.0782,  0.4451],
        [ 0.2784,  0.0853]], grad_fn=<PermuteBackward0>)


In [109]:
keys

tensor([[-0.2905, -0.1235],
        [ 0.1682, -0.4501],
        [ 0.5259,  0.5426],
        [ 1.0793,  0.7414],
        [-0.0488,  0.0919],
        [-0.5481, -0.5624]], grad_fn=<MmBackward0>)

In [110]:
queries

tensor([[ 1.6305,  0.7759],
        [-0.3151,  0.4547],
        [ 1.1107,  0.2536],
        [-0.1910, -0.2859],
        [-0.0482, -0.1574],
        [ 0.2847,  0.4491]], grad_fn=<MmBackward0>)

In [111]:
values

tensor([[ 0.7997,  0.4624],
        [ 0.4284, -0.2812],
        [ 0.5721,  0.7131],
        [ 0.1264,  0.4416],
        [-0.1656,  0.0160],
        [ 0.2324, -0.2417]], grad_fn=<MmBackward0>)

In [112]:
key_2 = keys[1]
attn_score_22 = query_2.dot(key_2)
attn_score_22

tensor(-0.2577, grad_fn=<DotBackward0>)

In [113]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([ 0.0354, -0.2577,  0.0810, -0.0030,  0.0571, -0.0830],
       grad_fn=<SqueezeBackward4>)


In [114]:
attn_score = queries @ keys.T

In [115]:
attn_score

tensor([[-0.5695, -0.0750,  1.2785,  2.3351, -0.0082, -1.3301],
        [ 0.0354, -0.2577,  0.0810, -0.0030,  0.0571, -0.0830],
        [-0.3540,  0.0727,  0.7217,  1.3868, -0.0309, -0.7514],
        [ 0.0908,  0.0965, -0.2556, -0.4181, -0.0169,  0.2655],
        [ 0.0334,  0.0628, -0.1108, -0.1687, -0.0121,  0.1149],
        [-0.1382, -0.1543,  0.3934,  0.6402,  0.0274, -0.4086]],
       grad_fn=<MmBackward0>)

In [116]:
dim_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / dim_k**0.5, dim=-1)
attn_weights_2

tensor([0.1738, 0.1413, 0.1795, 0.1691, 0.1765, 0.1598],
       grad_fn=<SoftmaxBackward0>)

In [117]:
class SelfAttention(torch.nn.Module):
    def __init__(self,d_in,d_out,qkv = False):
        super().__init__() 
        self.d_in = d_in
        self.d_out = d_out
        self.Wq = torch.nn.Linear(self.d_in,self.d_out,bias=qkv)
        self.Wk = torch.nn.Linear(self.d_in,self.d_out,bias=qkv)
        self.Wv = torch.nn.Linear(self.d_in,self.d_out,bias=qkv)

    def forward(self,x):
        query = self.Wq(x)
        key = self.Wk(x)
        value = self.Wv(x)

        dim_k = key.shape[-1]
        attn_score = query@key.T
        attn_weight = torch.softmax(attn_score / (dim_k**0.5), dim=-1)

        context_vec = attn_weight@value
        return context_vec
    

In [118]:
torch.manual_seed(789)
sa = SelfAttention(d_in, d_out)
print(sa(inputs))

tensor([[-0.0261,  0.0962],
        [ 0.0325,  0.1975],
        [-0.0152,  0.1091],
        [ 0.0179,  0.1680],
        [ 0.0112,  0.1753],
        [ 0.0100,  0.1716]], grad_fn=<MmBackward0>)
